# Experimento: ¿el momentum agrega algo al modelo?

**Base:** el modelo actual (Elo + forma por rol), con todo ajustado **solo con 2024**. Encima se prueban:

| Variable | Qué mide |
|---|---|
| `d_over` | Sorpresa acumulada en las últimas 10 partidas. Sorpresa = resultado − probabilidad que daba el Elo |
| `d_trend` | Tendencia del Elo: la misma sorpresa, pero en los últimos 21 días |
| `d_tourney` | Sorpresa acumulada dentro del torneo internacional en curso. Equivale a darle un K más alto a esas partidas |
| `x_prev` | Momentum dentro de la serie: +1 si el lado azul ganó la partida anterior, −1 si la perdió |

**Regla (fijada antes de mirar):** se adopta si mejora todo 2025 con un IC 95% que no cruce el 0 **y** no empeora ni en Worlds 2025 ni en el MSI 2026.

Un detalle importante: `d_over` y `d_trend` son casi lo mismo. El Elo cambia K × sorpresa después de cada partida, así que la pendiente del rating **es** la sorpresa acumulada. Solo cambia la ventana: 10 partidas contra 21 días.

In [1]:
import sys
from math import exp, log
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent))

import numpy as np
import pandas as pd
from scipy.optimize import minimize

from models.backtest import tune
from models.elo import run_elo, series_probability, team_regions
from models.features import ROLE_FEATURES, build_dataset, fit_norms
from models.metrics import accuracy, bootstrap_brier_diff, brier_score, log_loss
from models.momentum import FEATURES, momentum_features, series_probability_momentum
from models.oracles import load_season
from models.role_form import fit as fit_roles

OE_DIR = Path.cwd().parent / "data" / "raw" / "oracles_elixir"
YEARS = (2024, 2025, 2026)
games = {y: load_season(OE_DIR / f"{y}.csv") for y in YEARS}
regions = {y: team_regions(games[y]) for y in YEARS}
params, _ = tune([(games[2024], regions[2024])])
runs = {y: run_elo(games[y], params, regions[y]) for y in YEARS}
norms = fit_norms(games[2024])
weights, scales = fit_roles([runs[2024].records], norms)
print(params)
print("pesos de forma por rol (2024):", weights)

Params(k=15.0, region_offsets={'KR': 350.0, 'CN': 350.0, 'EU': 200.0, 'NA': 200.0})
pesos de forma por rol (2024): {'top': 0.2345, 'jng': 0.0899, 'mid': 0.1053, 'bot': 0.1919, 'sup': 0.1688}


## 1. Armar las filas

Cada fila tiene:
- el **log-odds base**: Elo + ajuste por forma de los jugadores, con los pesos de 2024;
- las cuatro variables de momentum, armadas solo con partidas anteriores (hay tests que lo verifican).

In [2]:
def dataset(year):
    rows = build_dataset(runs[year].records, norms)
    moms = momentum_features(runs[year].records)
    out = []
    for r, m, rec in zip(rows, moms, runs[year].records):
        role_adj = sum(weights[c[2:]] * r[c] / scales[c[2:]] for c in ROLE_FEATURES)
        out.append({**m, "base": r["elo_logit"] + role_adj, "blue_won": r["blue_won"], "page": r["page"],
                    "match_id": r["match_id"], "blue": r["blue"], "red": r["red"], "game": rec["game"]})
    return pd.DataFrame(out)

data = {y: dataset(y) for y in YEARS}
d24 = data[2024]
print("correlación d_over / d_trend (2024):", round(d24.d_over.corr(d24.d_trend), 3))
d24[list(FEATURES)].describe().round(3)

correlación d_over / d_trend (2024): 0.858


,d_over,d_trend,d_tourney,x_prev
count,3087.000,3087.000,3087.000,3087.000
mean,-0.120,-0.158,-0.026,-0.213
std,2.311,2.186,0.447,0.700
min,-9.117,-8.151,-5.274,-1.000
25%,-1.604,-1.479,0.000,-1.000
50%,-0.063,-0.043,0.000,0.000
75%,1.367,1.046,0.000,0.000
max,6.792,8.850,3.550,1.000


## 3. Ajustar los pesos (con 2024)

Regresión logística con el log-odds base como *offset*: el modelo actual queda fijo, y solo se aprende el término independiente (ventaja del lado azul) más el peso de cada variable de momentum.

La comparación justa es contra **"Base"**: el mismo procedimiento sin ninguna variable de momentum, que solo ajusta la ventaja del azul.

In [3]:
def fit_offset(df, cols, lam=0.001):
    X = df[list(cols)].to_numpy() if cols else np.zeros((len(df), 0))
    scale = X.std(axis=0) if cols else np.zeros(0)
    Z = X / scale if cols else X
    off, y = df.base.to_numpy(), df.blue_won.to_numpy()

    def loss(w):
        z = off + w[0] + Z @ w[1:]
        return np.mean(np.logaddexp(0, z) - y * z) + lam * np.sum(w[1:] ** 2)

    w = minimize(loss, np.zeros(len(cols) + 1), method="BFGS").x
    def predict(d):
        Xd = d[list(cols)].to_numpy() / scale if cols else np.zeros((len(d), 0))
        return 1 / (1 + np.exp(-(d.base.to_numpy() + w[0] + Xd @ w[1:])))
    return predict, dict(zip(("lado azul",) + tuple(cols), np.round(w, 4)))

VARIANTS = {"Base": (), "d_over": ("d_over",), "d_trend": ("d_trend",), "d_tourney": ("d_tourney",),
            "x_prev": ("x_prev",), "todas": FEATURES}
fitted = {name: fit_offset(d24, cols) for name, cols in VARIANTS.items()}
pd.DataFrame({name: w for name, (_, w) in fitted.items()}).T

,lado azul,d_over,d_trend,d_tourney,x_prev
Base,0.1544,NaN,NaN,NaN,NaN
d_over,0.1608,0.0891,NaN,NaN,NaN
d_trend,0.1636,NaN,0.1009,NaN,NaN
d_tourney,0.1527,NaN,NaN,-0.0330,NaN
x_prev,0.1898,NaN,NaN,NaN,0.1128
todas,0.1880,0.0110,0.0729,-0.0538,0.0926


**Pesos estandarizados:** cuánto cambia el log-odds por cada desvío estándar de la variable, *después* de controlar por el Elo y la forma.

- Positivo = momentum real: el Elo reacciona tarde.
- Cerca de 0 = el Elo y la forma ya capturan todo.
- Negativo = "regresión a la media": lo que sobresalió de más tiende a volver.

## 3b. Momentum en las series: primero mirar los datos

Si las partidas fueran independientes, el ganador de la partida anterior debería ganar la siguiente con la probabilidad que le da el modelo, ni más ni menos.

Un cuidado importante: en la mayoría de las ligas **el perdedor elige lado** y suele elegir azul. Si no se separa por lado, el efecto del lado se confunde con el momentum. Por eso la tabla está separada por el lado que le tocó al ganador anterior.

In [4]:
def series_table(df):
    s = df[df.x_prev != 0].copy()
    p = fitted["Base"][0](s)  # incluye la ventaja del lado azul
    # Desde el punto de vista del ganador de la partida anterior
    s["ganador_anterior_lado"] = np.where(s.x_prev > 0, "azul", "rojo")
    s["esperado"] = np.where(s.x_prev > 0, p, 1 - p)
    s["real"] = np.where(s.x_prev > 0, s.blue_won, 1 - s.blue_won)
    return s.groupby("ganador_anterior_lado").agg(partidas=("real", "size"), esperado=("esperado", "mean"),
                                                  real=("real", "mean")).round(3)

pd.concat({y: series_table(data[y]) for y in YEARS})

partidas  esperado   real
     ganador_anterior_lado                           
2024 azul                        498     0.626  0.683
     rojo                       1154     0.541  0.562
2025 azul                        244     0.625  0.680
     rojo                       1678     0.526  0.522
2026 azul                       1432     0.603  0.607
     rojo                        500     0.524  0.534

In [5]:
TESTS = {
    "2025 todas": data[2025],
    "Worlds 2025": data[2025][data[2025].page == "WLDs"],
    "2026 todas": data[2026],
    "MSI 2026": data[2026][data[2026].page == "MSI"],
}
rows = []
for test, df in TESTS.items():
    y = df.blue_won.tolist()
    base = list(fitted["Base"][0](df))
    for name, (predict, _) in fitted.items():
        p = list(predict(df))
        ci = bootstrap_brier_diff(p, base, y, df.match_id.tolist(), reps=2000)
        rows.append({"prueba": test, "n": len(df), "variante": name, "brier": round(brier_score(p, y), 4),
                     "log_loss": round(log_loss(p, y), 4), "precisión": round(accuracy(p, y), 3),
                     "Δ vs base": ci["diff"], "IC 95%": f"[{ci['low']:+.4f}, {ci['high']:+.4f}]"})
results = pd.DataFrame(rows)
results

,prueba,n,variante,brier,log_loss,precisión,Δ vs base,IC 95%
0,2025 todas,3496,Base,0.2171,0.6237,0.648,0.000000,"[+0.0000, +0.0000]"
1,2025 todas,3496,d_over,0.2161,0.6213,0.653,-0.000975,"[-0.0015, -0.0005]"
2,2025 todas,3496,d_trend,0.2168,0.6229,0.650,-0.000291,"[-0.0009, +0.0003]"
3,2025 todas,3496,d_tourney,0.2172,0.6239,0.650,0.000071,"[-0.0002, +0.0003]"
4,2025 todas,3496,x_prev,0.2174,0.6244,0.658,0.000254,"[-0.0005, +0.0010]"
5,2025 todas,3496,todas,0.2171,0.6237,0.652,-0.000026,"[-0.0010, +0.0009]"
6,Worlds 2025,96,Base,0.2260,0.6387,0.625,0.000000,"[+0.0000, +0.0000]"
7,Worlds 2025,96,d_over,0.2263,0.6392,0.646,0.000298,"[-0.0032, +0.0035]"
8,Worlds 2025,96,d_trend,0.2266,0.6399,0.625,0.000519,"[-0.0037, +0.0046]"
9,Worlds 2025,96,d_tourney,0.2264,0.6405,0.646,0.000402,"[-0.0041, +0.0058]"


## 4. Series con momentum

Si el peso de `x_prev` no es 0, la conversión de partida a serie ya no puede suponer partidas independientes. `series_probability_momentum` recorre todos los caminos de la serie: después de ganar, los log-odds suben w; después de perder, bajan w.

Un dato contraintuitivo: el momentum positivo **no** ayuda al favorito. Ganar arrastra a ganar y perder arrastra a perder, así que la serie se parece más a una sola partida, y la ventaja que da jugar al mejor de 5 se achica. Con momentum negativo pasa lo contrario.

In [6]:
_, w_prev = fitted["x_prev"]
w = w_prev["x_prev"] / d24.x_prev.std()  # peso por unidad de x_prev (±1)
print("w por partida ganada:", round(w, 4))

def series_eval(df):
    first = df.groupby("match_id").head(1).set_index("match_id")
    winners = {}  # serie -> {equipo: games ganados}
    for mid, b, r, won in zip(df.match_id, df.blue, df.red, df.blue_won):
        team = b if won == 1 else r
        winners.setdefault(mid, {}).setdefault(team, 0)
        winners[mid][team] += 1
    p_ind, p_mom, outcome = [], [], []
    for mid, row in first.iterrows():
        winner, n = max(winners[mid].items(), key=lambda kv: kv[1])
        best_of = 2 * n - 1
        z = row.base  # log-odds del primer game para el equipo azul de ese game
        p_ind.append(series_probability(1 / (1 + exp(-z)), best_of))
        p_mom.append(series_probability_momentum(z, w, best_of))
        outcome.append(1.0 if winner == row.blue else 0.0)
    return {"series": len(outcome), "brier independientes": round(brier_score(p_ind, outcome), 4),
            "brier con momentum": round(brier_score(p_mom, outcome), 4)}

pd.DataFrame({"Worlds 2025": series_eval(TESTS["Worlds 2025"]), "MSI 2026": series_eval(TESTS["MSI 2026"])}).T

w por partida ganada: 0.1611


,series,brier independientes,brier con momentum
Worlds 2025,44.0,0.1949,0.1948
MSI 2026,20.0,0.1737,0.1737


## 5. Conclusión

**Ninguna variable de momentum pasa la regla. El modelo queda como está (`elo-2`).**

- **`d_over` (sorpresa en las últimas 10 partidas):** es la única que mejora todo 2025 con un IC que no cruza el 0. Pero la mejora es mínima, 0.001 de Brier (menos del 0.5%), y en Worlds 2025 empeora un poco. Por la regla, no entra.
  - Tiene peso positivo: hay algo de momentum que el Elo y la forma tardan en captar, pero es muy chico.
- **`d_trend` (tendencia del Elo):** se correlaciona 0.86 con `d_over`, como esperábamos, y rinde menos. Contar por partidas funciona mejor que contar por días.
- **`d_tourney` (momentum dentro del torneo):** el peso sale **negativo** y no mejora nada. No hay evidencia de que un equipo que sorprende en la primera fase siga sorprendiendo.
  - Más bien lo contrario, aunque débil: **regresión a la media**. Lo que sobresalió por suerte tiende a volver.
- **`x_prev` (momentum dentro de la serie):** en 2024 aprende un peso de +0.16 por partida ganada, pero fuera de la muestra no mejora ni las partidas ni las series de Worlds 2025 y del MSI 2026 (el Brier de las series casi no cambia).
  - La tabla 3b muestra por qué. En 2024 y 2025, cuando al ganador le tocaba azul, ganaba ~6 puntos más de lo esperado; con rojo, casi lo esperado.
  - En 2026, con muchas más partidas en esa situación (1432), el efecto desaparece: 60.7% real contra 60.3% esperado. Un patrón que aparece en unos años y en otros no suele ser ruido o un cambio de reglas, no una ley del juego.
  - El supuesto de partidas independientes, aunque no es exacto, alcanza para predecir.
- **Lección:** el Elo ya es un promedio con memoria que se actualiza después de cada partida. La mayoría de las ideas de "momentum" ya están adentro. Lo que queda afuera es tan chico que no sobrevive a la prueba fuera de la muestra.